In [1]:
from langchain.tools import tool
# 创建一个工具最简单的方式是使用@tool装饰器。默认情况下，函数的文档字符串将成为该工具的描述，帮助模型理解何时使用它：类型提示是必需的，因为它们定义了工具的输入方案。文档字符串应具有信息性和简洁性，以帮助模型理解工具的目的。
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [2]:
# 默认的工具名称来自函数名称，你可以自定义工具名称
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

web_search


In [3]:
# 覆盖自动生成的工具描述，以提供更清晰的模型指导：
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [4]:
# 使用 Pydantic 模型或 JSON 架构定义复杂输入  你给工具函数定义参数时，有两个名字你不能用：config 和 runtime。 因为 LangChain 内部已经占用了这两个名字，它们有特殊用途。
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [5]:
from setuptools import Command
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import ToolRuntime


# Access context（访问上下文）
# 1. State（状态）—— 短期记忆
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """获取用户最近说的一句话"""
    messages = runtime.state["messages"]  # 拿到所有聊天记录
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content
    return "没找到"

@tool
def set_user_name(new_name: str) -> Command:
    """把用户名字存到状态里"""
    return Command(update={"user_name": new_name})

In [ ]:
# Context（上下文）—— 身份信息 是只读的，不能改 通常放用户 ID、会话信息这类东西

from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str  # 定义上下文里有什么

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """查当前用户的账户信息"""
    user_id = runtime.context.user_id  # 拿到用户ID
    # 然后用这个ID去数据库查...
    return f"用户 {user_id} 的账户信息..."
#
# 调用 agent 的时候传入上下文：
# result = agent.invoke(
#     {"messages": [...]},
#     context=UserContext(user_id="user123")  # 这里告诉系统"当前用户是谁"
# )